# Olist Brazilian E-Commerce Data Analysis

This project analyzes the Olist Brazilian e-commerce dataset to understand
sales performance, customer behavior, product performance, delivery
efficiency, and customer satisfaction.

The purpose of this notebook is to perform initial data understanding and
data-quality assessment before data cleaning and downstream business
analysis.

The analysis focuses on:

- Dataset structure and table relationships
- Missing values and duplicate records
- Primary and foreign key integrity
- Data types and value validity
- Analytical grain of each dataset
- Data-cleaning decisions for downstream analysis

## 1. Environment Setup

Import the required Python libraries and configure the notebook display
settings.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Load Raw Datasets

Load the nine raw CSV files from the Olist e-commerce dataset.

Each file represents a different business entity or transactional component,
including customers, orders, order items, payments, reviews, products,
sellers, product-category translations, and geolocation data.

The datasets are loaded without modification at this stage so that their
original structure and data quality can be assessed before cleaning.

In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

### 2.1 Dataset Overview

Create a high-level summary of all nine datasets to understand their
overall size and initial data-quality characteristics.

For each dataset, the following information is examined:

- Number of rows
- Number of columns
- Number of exact duplicate records
- Total number of missing values

This provides an initial overview of the datasets and helps identify areas
that require further investigation.

In [3]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

summary = []

for name, df in datasets.items():
    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicates": df.duplicated().sum(),
        "missing_values": df.isnull().sum().sum()
    })

summary_df = pd.DataFrame(summary)
summary_df

,dataset,rows,columns,duplicates,missing_values
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,0,145903
5,orders,99441,8,0,4908
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


### 2.2 Dataset Schema and Column Structure

Inspect the column names of each dataset to understand the available
variables and the overall relational structure of the data.

This step helps identify:

- Potential primary and foreign keys
- Customer, order, product, seller, payment, and review attributes
- Date and timestamp fields
- Geographic variables
- Fields that may be required for downstream joins and analysis

Detailed key relationships and data types will be validated in later
sections.

In [4]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print(df.columns.tolist())


===== customers =====
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

===== geolocation =====
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

===== order_items =====
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

===== payments =====
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

===== reviews =====
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

===== orders =====
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

===== products =====
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'prod

### 2.3 Initial Inspection of the Orders Dataset

Inspect the structure of the orders dataset, which serves as the central
transaction table connecting customers with order items, payments, and
reviews.

The inspection focuses on:

- Number of records and columns
- Column data types
- Non-null value counts
- Initial identification of missing values
- Timestamp fields that may require type conversion

Detailed missing-value patterns and data-type issues will be investigated
in later sections.

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


#### Initial Orders Dataset Findings

The orders dataset contains 99,441 records and 8 columns.

Initial inspection shows that:

- `order_id`, `customer_id`, `order_status`, and `order_purchase_timestamp`
  contain no missing values.
- `order_approved_at`, `order_delivered_carrier_date`, and
  `order_delivered_customer_date` contain missing values.
- All five order-related timestamp fields are currently stored as strings
  rather than datetime values.
- `order_id` and `customer_id` are also stored as strings, which is
  appropriate for identifier fields.

The missing timestamp values and their relationship with order lifecycle
status will be investigated in the missing-value analysis.

Timestamp fields will be evaluated for datetime conversion during the
data-type assessment and converted during the cleaning stage.

#### Sample Order Records

Preview the first five records to examine the actual values and structure
of the orders dataset.

In [6]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 3. Missing Value Analysis

Missing values are examined to determine whether they represent data-quality
issues, optional information, or expected missingness caused by business
processes and order lifecycle stages.

The analysis begins with the orders dataset because several lifecycle
timestamp fields were identified as incomplete during the initial inspection.

### 3.1 Missing Values in the Orders Dataset

Calculate the number of missing values in each order field to identify
which lifecycle timestamps require further investigation.

In [7]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

#### Initial Missing-Value Findings

Missing values are concentrated in three order lifecycle timestamps:

- `order_approved_at`: 160 missing values
- `order_delivered_carrier_date`: 1,783 missing values
- `order_delivered_customer_date`: 2,965 missing values

The presence of missing lifecycle timestamps does not necessarily indicate
data-quality errors. For example, canceled, unavailable, or undelivered
orders may legitimately lack approval, carrier, or customer delivery
timestamps.

The relationship between missing timestamps and `order_status` will
therefore be investigated before determining any cleaning action.

### 3.2 Order Status Distribution

Examine the distribution of order statuses to understand the different
stages of the order lifecycle.

Order status is particularly important for interpreting missing lifecycle
timestamps because orders that were canceled, unavailable, shipped, or
still processing may not have reached the same stages as completed orders.

In [8]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

#### Order Status Findings

The majority of orders are marked as `delivered`, with 96,478 completed
orders.

A smaller number of orders remain in other lifecycle stages, including
`shipped`, `invoiced`, `processing`, `created`, and `approved`. The dataset
also contains `canceled` and `unavailable` orders.

These different order statuses provide an important context for interpreting
missing lifecycle timestamps. For example, an order that was canceled or
never progressed to delivery would not necessarily be expected to contain
carrier or customer delivery timestamps.

Therefore, missing order timestamps should be evaluated together with
`order_status` rather than treated uniformly as data-quality errors.

### 3.3 Missing Values Across All Datasets

Extend the missing-value assessment to all nine datasets and calculate both
the number and percentage of missing values for each affected column.

Missing-value percentages are included because the datasets differ
substantially in size, making raw missing counts alone insufficient for
comparing the relative severity of missingness across tables.

Only columns containing at least one missing value are included in the
summary.

In [9]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_summary.append({
                "dataset": name,
                "column": column,
                "missing_count": missing_count,
                "missing_pct": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_df = (
    pd.DataFrame(missing_summary)
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

missing_df

,dataset,column,missing_count,missing_pct
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


#### Cross-Dataset Missing-Value Findings

The missing-value summary reveals several distinct patterns across the
datasets:

- Review text fields contain the highest proportions of missing values.
  `review_comment_title` is missing for 88.34% of review records, while
  `review_comment_message` is missing for 58.70%.
- Missing values in the orders dataset are concentrated in lifecycle
  timestamps, with `order_delivered_customer_date` having the highest
  missing rate among order fields.
- Several product attributes share the same number and percentage of
  missing values, suggesting that the missingness may affect the same
  group of products.
- Missing values in physical product dimensions are extremely rare.

These patterns require different interpretations and should not be handled
with a single blanket missing-value strategy. Further investigation will
distinguish optional information, structural missingness, and potential
data-quality issues.

### 3.4 Product Missing-Value Pattern Investigation

Investigate whether products with missing category information also have
missing values in other descriptive attributes.

This helps determine whether the missing product attributes occur
independently or represent a systematic missing-data pattern affecting the
same group of products.

In [10]:
product_missing_core = products[
    products["product_category_name"].isna()
]

product_missing_core[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isna().sum()

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64

#### Product Missing-Value Findings

All 610 products with missing `product_category_name` also have missing
values for `product_name_lenght`, `product_description_lenght`, and
`product_photos_qty`.

This indicates a systematic missing-data pattern rather than independent
missing values across individual product attributes.

These products should not be removed solely because their descriptive
metadata is incomplete. Their transaction activity will be examined before
determining the appropriate cleaning strategy.

#### Missing Product Dimensions

Inspect products with missing physical attributes, including weight, length,
height, and width.

Unlike descriptive metadata, these attributes may be relevant to logistics,
freight, and product-level analysis. Identifying the affected products helps
determine whether the missing values are isolated cases or part of a broader
data-quality pattern.

In [11]:
products[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Product Dimension Findings

Only two products contain missing values in their physical attributes,
indicating that missing product dimensions are isolated rather than
widespread across the product dataset.

Because these products may still be associated with valid transactions,
they should not be removed solely because their physical specifications are
incomplete.

Their transaction activity should be checked before determining how the
missing dimension values will be handled during data cleaning.

#### Transaction Activity of Products with Missing Dimensions

Check whether products with missing physical dimensions appear in the
`order_items` dataset.

This step determines whether the affected products are associated with
actual transactions. If valid transactions exist, removing these products
solely because of incomplete physical attributes could result in the loss
of legitimate sales records.

In [12]:
missing_dimension_product_ids = products.loc[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1),
    "product_id"
]

order_items[
    order_items["product_id"].isin(
        missing_dimension_product_ids
    )
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
7098,101157d4fae1c9fb74a00a5dee265c25,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-11 08:02:26,29.0,14.52
9233,1521c6bb7b1028154c8c67cf80fa809f,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:10:16,29.0,16.05
28715,415cfaaaa8cea49f934470548797fed1,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:35:19,29.0,14.52
28716,415cfaaaa8cea49f934470548797fed1,2,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:35:19,29.0,14.52
39299,595316a07cd3dea9db7adfcc7e247ae7,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-08-18 04:26:04,39.0,9.27
48424,6e150190fbe04c642a9cf0b80d83ee16,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-06-30 16:45:14,39.0,16.79
48980,6f497c40431d5fb0cfbd6c943dd29215,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-11 05:55:32,29.0,10.96
58833,85f8ad45e067abd694b627859fa57453,1,09ff539a621711667c43eba6a3bd8466,8b8cfc8305aa441e4239358c9f6f2485,2017-02-03 21:40:02,1934.0,27.00
71134,a2456e7f02197951664897a94c87242d,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-06 11:50:09,29.0,24.84
73556,a7a43f469c0d7bdb0a23a82db125aefa,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-08-28 13:15:11,39.0,15.10


#### Transaction Validation Findings

The products with missing physical dimensions are associated with valid
records in the `order_items` dataset and contain positive transaction
values.

This indicates that the missing dimensions are product master-data issues
rather than invalid transaction records.

Therefore, the affected products and their transactions should be retained.
Missing physical attributes should be handled separately during the data
cleaning stage rather than removing the associated sales records.

### 3.5 Order Timestamp Missingness by Status

Investigate whether missing order lifecycle timestamps are associated with
specific order statuses.

For each order status, calculate the total number of orders and the number
of missing values in the approval, carrier delivery, and customer delivery
timestamps.

This helps distinguish expected lifecycle-related missingness from potential
data-quality issues.

In [13]:
missing_timestamp_by_status = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),

        missing_approved=(
            "order_approved_at",
            lambda x: x.isna().sum()
        ),

        missing_carrier_date=(
            "order_delivered_carrier_date",
            lambda x: x.isna().sum()
        ),

        missing_customer_date=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        )
    )
)

missing_timestamp_by_status

,total_orders,missing_approved,missing_carrier_date,missing_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


#### Order Timestamp Missingness Findings

Missing order timestamps are strongly associated with order lifecycle status.

- `created`, `approved`, `invoiced`, `processing`, and `unavailable` orders
  generally lack delivery timestamps because they did not reach the
  corresponding delivery stages.
- All 1,107 `shipped` orders are missing
  `order_delivered_customer_date`, which is consistent with orders that
  have been shipped but not yet recorded as delivered.
- Most `canceled` orders also lack carrier and customer delivery timestamps,
  although some contain partial lifecycle information.
- `delivered` orders are largely complete, but a small number still contain
  missing lifecycle timestamps: 14 missing approval timestamps, 2 missing
  carrier delivery timestamps, and 8 missing customer delivery timestamps.

Overall, most missing timestamps appear to be structurally related to the
order lifecycle rather than general data-quality failures. However, missing
timestamps among delivered orders require further investigation because
these orders are expected to have completed the delivery process.

#### Delivered Orders with Missing Lifecycle Timestamps

Because delivered orders are expected to have completed the order lifecycle,
further investigate delivered orders that are still missing approval,
carrier delivery, or customer delivery timestamps.

These records are examined separately to distinguish potential data-quality
issues from the structural missingness observed in incomplete order statuses.

In [14]:
delivered_missing_timestamps = orders[
    (orders["order_status"] == "delivered")
    &
    (
        orders["order_approved_at"].isna()
        | orders["order_delivered_carrier_date"].isna()
        | orders["order_delivered_customer_date"].isna()
    )
]

print(
    "Delivered orders with missing timestamps:",
    len(delivered_missing_timestamps)
)

delivered_missing_timestamps

Delivered orders with missing timestamps: 23


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


#### Delivered Order Findings

A total of 23 delivered orders contain at least one missing lifecycle
timestamp.

Across these 23 orders, there are 24 missing timestamp cells:

- 14 missing `order_approved_at`
- 2 missing `order_delivered_carrier_date`
- 8 missing `order_delivered_customer_date`

Because there are 24 missing timestamp cells across 23 orders, at least one
order is missing more than one lifecycle timestamp.

These cases represent only a very small portion of delivered orders and do
not justify removing the entire records. Instead, the affected orders should
be retained and excluded only from analyses that specifically require the
missing timestamp.

#### Missing Timestamp Patterns in Delivered Orders

Examine the combinations of missing lifecycle timestamps among delivered
orders.

This step identifies whether the affected orders are missing only one
timestamp or multiple timestamps simultaneously, providing a clearer view
of the anomaly pattern within completed orders.

In [15]:
delivered_missing_pattern = (
    delivered_missing_timestamps
    .assign(
        missing_approved=lambda x:
            x["order_approved_at"].isna(),

        missing_carrier=lambda x:
            x["order_delivered_carrier_date"].isna(),

        missing_customer=lambda x:
            x["order_delivered_customer_date"].isna()
    )
    .groupby(
        [
            "missing_approved",
            "missing_carrier",
            "missing_customer"
        ]
    )
    .size()
    .reset_index(name="order_count")
)

delivered_missing_pattern

,missing_approved,missing_carrier,missing_customer,order_count
0,False,False,True,7
1,False,True,False,1
2,False,True,True,1
3,True,False,False,14


#### Missing Timestamp Pattern Findings

The 23 delivered orders with incomplete lifecycle timestamps are primarily
single-field missing cases.

Most affected orders are missing only the approval timestamp, while several
orders are missing only the customer delivery timestamp. A very small number
of orders involve missing carrier timestamps, including one order where both
carrier and customer delivery timestamps are missing.

This confirms that the 24 missing timestamp cells are distributed across
23 orders rather than representing 24 separate orders.

The affected records should be retained, with each missing timestamp handled
according to the requirements of downstream analyses.

## 4. Duplicate Analysis

Duplicate records are examined across all nine datasets to identify potential
data redundancy and determine whether repeated rows represent true duplicates
or observations occurring at different levels of granularity.

For each dataset, the analysis calculates:

- Total number of records
- Number of exact duplicate rows
- Percentage of exact duplicate rows

Datasets containing duplicates will be investigated further before any
records are removed.

In [16]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "dataset": name,
        "total_rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_pct": round(
            duplicate_count / len(df) * 100, 2
        )
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df

,dataset,total_rows,duplicate_rows,duplicate_pct
0,customers,99441,0,0.00
1,geolocation,1000163,261831,26.18
2,order_items,112650,0,0.00
3,payments,103886,0,0.00
4,reviews,99224,0,0.00
5,orders,99441,0,0.00
6,products,32951,0,0.00
7,sellers,3095,0,0.00
8,category_translation,71,0,0.00


### 4.1 Exact Duplicate Summary

Exact duplicate rows are found only in the `geolocation` dataset.

The dataset contains 261,831 duplicate rows out of 1,000,163 total records,
representing approximately 26.18% of the dataset.

No exact duplicate rows are identified in the other eight datasets.

Because the `geolocation` table may contain multiple observations for the
same ZIP code prefix, the duplicated records require further investigation
before any cleaning decision is made.

### 4.2 Geolocation Duplicate Investigation

Inspect the duplicated records in the `geolocation` dataset to understand
the nature of the duplication.

All occurrences of duplicated rows are included and sorted by ZIP code
prefix, allowing repeated geographic observations to be compared directly.

This helps determine whether the duplicates are exact repeated records or
whether the same ZIP code prefix can also contain different geographic
observations.

In [17]:
geolocation[
    geolocation.duplicated(keep=False)
].sort_values(
    "geolocation_zip_code_prefix"
).head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1004,1001,-23.549292,-46.633559,sao paulo,SP
771,1001,-23.550498,-46.634338,sao paulo,SP
1435,1001,-23.549292,-46.633559,sao paulo,SP
912,1001,-23.550498,-46.634338,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
99,1001,-23.549292,-46.633559,sao paulo,SP
851,1001,-23.549825,-46.633970,sao paulo,SP
639,1001,-23.550498,-46.634338,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
1246,1001,-23.549292,-46.633559,sao paulo,SP


#### Geolocation ZIP Code Granularity

Compare the total number of geolocation records with the number of unique
ZIP code prefixes to assess the granularity of the raw geolocation dataset.

A substantial difference between these values would indicate that multiple
geographic observations are recorded for the same ZIP code prefix.

In [18]:
print(
    "Total rows:",
    len(geolocation)
)

print(
    "Unique ZIP codes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

Total rows: 1000163
Unique ZIP codes: 19015


#### Observations per ZIP Code Prefix

Count the number of geolocation observations associated with each ZIP code
prefix and identify the ZIP prefixes with the highest number of records.

This helps confirm that the raw geolocation dataset operates at the
geographic-observation level rather than containing a single record for
each ZIP code prefix.

In [19]:
geo_zip_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .size()
    .sort_values(ascending=False)
)

geo_zip_counts.head(10)

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
dtype: int64

#### Distribution of Observations per ZIP Code Prefix

Summarize the distribution of geolocation observations per ZIP code prefix
using the mean, median, and maximum number of records.

This provides additional context on whether geographic observations are
evenly distributed across ZIP prefixes or concentrated in a smaller number
of locations.

In [20]:
print(
    "Average observations per ZIP code:",
    round(geo_zip_counts.mean(), 2)
)

print(
    "Median observations per ZIP code:",
    geo_zip_counts.median()
)

print(
    "Maximum observations for a ZIP code:",
    geo_zip_counts.max()
)

Average observations per ZIP code: 52.6
Median observations per ZIP code: 29.0
Maximum observations for a ZIP code: 1146


#### Geolocation Granularity Findings

The raw `geolocation` dataset contains 1,000,163 geographic observations
across 19,015 unique ZIP code prefixes.

Each ZIP code prefix is associated with multiple observations. On average,
a ZIP prefix contains 52.6 records, while the median is 29 records. The
largest ZIP prefix contains 1,146 observations, indicating that geographic
observations are unevenly distributed across ZIP prefixes.

Inspection of duplicated records also shows that the dataset contains both:

- Exact duplicate geographic observations
- Multiple observations for the same ZIP code prefix with different
  latitude and longitude values

Therefore, the raw analytical grain of the `geolocation` dataset is a
geographic observation rather than one record per ZIP code prefix.

The raw table should not be joined directly to customer or seller records
using ZIP code prefix, as this could create duplicate rows and inflate
downstream analytical results.

During data cleaning, exact duplicate observations will be removed and the
geolocation data will be transformed to ZIP-code-level granularity before
being used in geographic analysis.

## 5. Primary Key and Composite Key Analysis

Primary and composite keys are examined to verify whether each dataset has
a reliable identifier that uniquely represents its analytical grain.

A valid primary key should:

- Uniquely identify each record
- Contain no duplicate values
- Contain no missing values

The analysis begins with datasets that are expected to contain a single
primary key. Transactional tables with potentially multiple records per
business entity will be evaluated separately using composite keys.

### 5.1 Primary Key Validation

Validate the candidate primary keys for the customer, order, product,
seller, and product-category translation datasets.

For each candidate key, compare the total number of records with the number
of unique values and check for duplicate or missing key values.

In [21]:
primary_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
    "category_translation": "product_category_name"
}

pk_summary = []

for name, key in primary_keys.items():
    df = datasets[name]

    pk_summary.append({
        "dataset": name,
        "candidate_key": key,
        "total_rows": len(df),
        "unique_values": df[key].nunique(),
        "duplicate_keys": df[key].duplicated().sum(),
        "missing_keys": df[key].isna().sum()
    })

pk_df = pd.DataFrame(pk_summary)

pk_df

,dataset,candidate_key,total_rows,unique_values,duplicate_keys,missing_keys
0,customers,customer_id,99441,99441,0,0
1,orders,order_id,99441,99441,0,0
2,products,product_id,32951,32951,0,0
3,sellers,seller_id,3095,3095,0,0
4,category_translation,product_category_name,71,71,0,0


#### Primary Key Findings

All five candidate primary keys satisfy the expected uniqueness and
completeness requirements.

For each dataset:

- The number of unique key values equals the total number of records.
- No duplicate key values are present.
- No missing key values are present.

Therefore, the following fields can be treated as valid primary keys:

- `customers.customer_id`
- `orders.order_id`
- `products.product_id`
- `sellers.seller_id`
- `category_translation.product_category_name`

These validated keys provide reliable identifiers for downstream joins and
relational integrity checks.

### 5.2 Customer Identifier Analysis

The customers dataset contains two customer identifiers:
`customer_id` and `customer_unique_id`.

Although `customer_id` has been validated as the primary key of the
customers table, `customer_unique_id` represents the identifier used to
recognize the same customer across multiple orders.

Compare the number of customer records, unique `customer_id` values, and
unique `customer_unique_id` values to clarify the role of each identifier
in downstream customer analysis.

In [22]:
print(
    "Total customer records:",
    len(customers)
)

print(
    "Unique customer_id:",
    customers["customer_id"].nunique()
)

print(
    "Unique customer_unique_id:",
    customers["customer_unique_id"].nunique()
)

Total customer records: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096


#### Customer Identifier Findings

The customers dataset contains 99,441 records and 99,441 unique
`customer_id` values, confirming that `customer_id` uniquely identifies
each customer record.

However, only 96,096 unique `customer_unique_id` values are present. This
indicates that some real-world customers are associated with multiple
`customer_id` values across different orders.

Therefore:

- `customer_id` should be used to join the `customers` and `orders` tables.
- `customer_unique_id` should be used for customer-level analysis, such as
  identifying repeat customers, purchase frequency, and RFM segmentation.

Distinguishing between these two identifiers is important to avoid
overestimating the number of unique customers in downstream analysis.

### 5.3 Order Items Composite Key Analysis

The `order_items` dataset may contain multiple products within the same
order, so `order_id` alone is not expected to uniquely identify each row.

Evaluate the combination of `order_id` and `order_item_id` to determine
whether it uniquely identifies individual order-item records.

In [23]:
print(
    "Total order item rows:",
    len(order_items)
)

print(
    "Unique order_id:",
    order_items["order_id"].nunique()
)

print(
    "Unique (order_id, order_item_id):",
    order_items[
        ["order_id", "order_item_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate composite keys:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Total order item rows: 112650
Unique order_id: 98666
Unique (order_id, order_item_id): 112650
Duplicate composite keys: 0


#### Order Items Key Findings

The `order_items` dataset contains 112,650 records associated with 98,666
unique orders.

Because the number of order-item records exceeds the number of unique
`order_id` values, a single order can contain multiple item records.
Therefore, `order_id` alone cannot uniquely identify each row in the
`order_items` dataset.

The combination of `order_id` and `order_item_id` produces 112,650 unique
combinations, equal to the total number of records, with no duplicate
composite keys.

Therefore, (`order_id`, `order_item_id`) can be treated as a valid composite
key for the `order_items` dataset.

### 5.4 Payments Composite Key Analysis

The `payments` dataset may contain multiple payment records for the same
order, so `order_id` alone may not uniquely identify each payment record.

Evaluate the combination of `order_id` and `payment_sequential` to determine
whether it uniquely identifies individual payment records.

In [24]:
print(
    "Total payment rows:",
    len(payments)
)

print(
    "Unique order_id:",
    payments["order_id"].nunique()
)

print(
    "Unique (order_id, payment_sequential):",
    payments[
        ["order_id", "payment_sequential"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate composite keys:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

Total payment rows: 103886
Unique order_id: 99440
Unique (order_id, payment_sequential): 103886
Duplicate composite keys: 0


#### Payments Key Findings

The `payments` dataset contains 103,886 payment records associated with
99,440 unique orders.

Because the number of payment records exceeds the number of unique
`order_id` values, some orders contain multiple payment records. Therefore,
`order_id` alone cannot uniquely identify each row in the `payments`
dataset.

The combination of `order_id` and `payment_sequential` produces 103,886
unique combinations, equal to the total number of records, with no duplicate
composite keys.

Therefore, (`order_id`, `payment_sequential`) can be treated as a valid
composite key for the `payments` dataset.

### 5.5 Reviews Key Analysis

The `reviews` dataset contains both `review_id` and `order_id`. Before
defining the appropriate key structure, each identifier is evaluated
individually for uniqueness.

If neither identifier uniquely identifies every review record, their
combination will be evaluated as a potential composite key.

In [25]:
print(
    "Total review rows:",
    len(reviews)
)

print(
    "Unique review_id:",
    reviews["review_id"].nunique()
)

print(
    "Duplicate review_id:",
    reviews["review_id"].duplicated().sum()
)

print(
    "Unique order_id:",
    reviews["order_id"].nunique()
)

print(
    "Duplicate order_id:",
    reviews["order_id"].duplicated().sum()
)

Total review rows: 99224
Unique review_id: 98410
Duplicate review_id: 814
Unique order_id: 98673
Duplicate order_id: 551


#### Duplicate Review ID Investigation

Because `review_id` is not unique, examine how frequently individual review
IDs appear in the dataset.

This helps determine whether duplicated review IDs are isolated cases or
whether some identifiers are associated with a larger number of records.

In [26]:
review_id_counts = reviews["review_id"].value_counts()

print(
    "Review IDs appearing more than once:",
    (review_id_counts > 1).sum()
)

print(
    "Maximum records for one review_id:",
    review_id_counts.max()
)

review_id_counts.head(10)

Review IDs appearing more than once: 789
Maximum records for one review_id: 3


review_id
c444278834184f72b1484dfe47de7f97    3
308316408775d1600dad81bd3184556d    3
2d6ac45f859465b5c185274a1c929637    3
3415c9f764e478409e8e0660ae816dd2    3
4219a80ab469e3fc9901437b73da3f75    3
e44840754f12fad2b8646712121b349a    3
ddc52555ca27b0fe67d5255147682d2d    3
7b606b0d57b078384f0b58eac1d41d78    3
4d0e6dd087008d1f992d25ef6e1f619f    3
08528f70f579f0c830189efc523d2182    3
Name: count, dtype: int64

#### Review Composite Key Validation

Because neither `review_id` nor `order_id` uniquely identifies every review
record, evaluate their combination as a potential composite key.

The combination should uniquely identify all review records without duplicate
key pairs if it is suitable as the dataset's composite key.

In [27]:
print(
    "Unique (review_id, order_id):",
    reviews[
        ["review_id", "order_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate composite keys:",
    reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

Unique (review_id, order_id): 99224
Duplicate composite keys: 0


#### Reviews Key Findings

The `reviews` dataset contains 99,224 records. Neither `review_id` nor
`order_id` is individually unique.

A total of 789 `review_id` values appear more than once, with a maximum of
three records associated with a single `review_id`. This confirms that
`review_id` alone cannot serve as a unique key.

Similarly, `order_id` is not unique because some orders are associated with
multiple review records.

However, the combination of `review_id` and `order_id` produces 99,224
unique combinations, equal to the total number of records, with no duplicate
composite keys.

Therefore, (`review_id`, `order_id`) can be treated as a valid composite key
for the `reviews` dataset.

This relationship should be considered carefully when review data is joined
or aggregated at the order level.

### 5.6 Key Structure Summary

The key validation confirms that the Olist datasets operate at different
levels of granularity.

Single-column primary keys are sufficient for the `customers`, `orders`,
`products`, `sellers`, and `category_translation` datasets. The
transactional `order_items`, `payments`, and `reviews` datasets require
composite keys to uniquely identify individual records.

The validated key structures are:

- `customers`: `customer_id`
- `orders`: `order_id`
- `products`: `product_id`
- `sellers`: `seller_id`
- `category_translation`: `product_category_name`
- `order_items`: (`order_id`, `order_item_id`)
- `payments`: (`order_id`, `payment_sequential`)
- `reviews`: (`review_id`, `order_id`)

Within the customers dataset, `customer_id` identifies individual customer
records and should be used to join customers with orders, while
`customer_unique_id` should be used to identify recurring real-world
customers in customer-level analysis.

The `geolocation` dataset does not have a single ZIP-code-level primary key
because multiple geographic observations may exist for the same ZIP code
prefix.

These key structures will guide the foreign-key integrity checks and help
prevent incorrect joins or unintended row multiplication in downstream
analysis.

## 6. Foreign Key Integrity

Foreign key relationships are validated to ensure that records in related
datasets can be reliably connected without referencing missing parent
records.

For each major relationship, foreign key values are checked against the
corresponding primary key in the parent dataset.

The following relationships are evaluated:

- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`
- `payments.order_id` → `orders.order_id`
- `reviews.order_id` → `orders.order_id`

Any foreign key value that does not exist in the corresponding parent table
is treated as an orphan record.

In [28]:
foreign_key_checks = [
    {
        "relationship": "orders → customers",
        "foreign_key": "customer_id",
        "orphan_count": (
            ~orders["customer_id"].isin(
                customers["customer_id"]
            )
        ).sum()
    },
    {
        "relationship": "order_items → orders",
        "foreign_key": "order_id",
        "orphan_count": (
            ~order_items["order_id"].isin(
                orders["order_id"]
            )
        ).sum()
    },
    {
        "relationship": "order_items → products",
        "foreign_key": "product_id",
        "orphan_count": (
            ~order_items["product_id"].isin(
                products["product_id"]
            )
        ).sum()
    },
    {
        "relationship": "order_items → sellers",
        "foreign_key": "seller_id",
        "orphan_count": (
            ~order_items["seller_id"].isin(
                sellers["seller_id"]
            )
        ).sum()
    },
    {
        "relationship": "payments → orders",
        "foreign_key": "order_id",
        "orphan_count": (
            ~payments["order_id"].isin(
                orders["order_id"]
            )
        ).sum()
    },
    {
        "relationship": "reviews → orders",
        "foreign_key": "order_id",
        "orphan_count": (
            ~reviews["order_id"].isin(
                orders["order_id"]
            )
        ).sum()
    }
]

fk_df = pd.DataFrame(foreign_key_checks)

fk_df

,relationship,foreign_key,orphan_count
0,orders → customers,customer_id,0
1,order_items → orders,order_id,0
2,order_items → products,product_id,0
3,order_items → sellers,seller_id,0
4,payments → orders,order_id,0
5,reviews → orders,order_id,0


### 6.1 Foreign Key Validation Findings

All six tested foreign key relationships contain zero orphan records.

This confirms that:

- Every `orders.customer_id` exists in the `customers` dataset.
- Every `order_items.order_id` exists in the `orders` dataset.
- Every `order_items.product_id` exists in the `products` dataset.
- Every `order_items.seller_id` exists in the `sellers` dataset.
- Every `payments.order_id` exists in the `orders` dataset.
- Every `reviews.order_id` exists in the `orders` dataset.

The core relational structure therefore shows strong referential integrity,
and no orphan records need to be removed or corrected for these
relationships.

However, foreign key integrity only confirms that existing child records
reference valid parent records. It does not guarantee that every parent
record has a corresponding record in each related table.

Relationship coverage will therefore be examined separately.

### 6.2 Order Relationship Coverage

The presence of related records is evaluated from the order perspective.

Although all child records reference valid orders, some orders may not have
corresponding order-item, payment, or review records. These cases are
investigated to determine whether they are associated with specific order
statuses or lifecycle conditions.

In [29]:
order_coverage = pd.DataFrame({
    "relationship": [
        "Orders with items",
        "Orders with payments",
        "Orders with reviews"
    ],
    "orders_with_record": [
        orders["order_id"].isin(
            order_items["order_id"]
        ).sum(),

        orders["order_id"].isin(
            payments["order_id"]
        ).sum(),

        orders["order_id"].isin(
            reviews["order_id"]
        ).sum()
    ]
})

order_coverage["total_orders"] = len(orders)

order_coverage["orders_without_record"] = (
    order_coverage["total_orders"]
    - order_coverage["orders_with_record"]
)

order_coverage["coverage_pct"] = (
    order_coverage["orders_with_record"]
    / order_coverage["total_orders"]
    * 100
).round(2)

order_coverage

,relationship,orders_with_record,total_orders,orders_without_record,coverage_pct
0,Orders with items,98666,99441,775,99.22
1,Orders with payments,99440,99441,1,100.00
2,Orders with reviews,98673,99441,768,99.23


#### Orders Without Order Items

A total of 775 orders do not have corresponding records in the
`order_items` dataset.

To determine whether these missing relationships are associated with
specific stages of the order lifecycle, examine the status distribution
of orders without order-item records.

In [30]:
orders_without_items = orders[
    ~orders["order_id"].isin(
        order_items["order_id"]
    )
]

orders_without_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

##### Orders Without Order Items Findings

Among the 775 orders without corresponding `order_items` records, 603 are
classified as `unavailable` and 164 as `canceled`.

Together, these two statuses account for 767 of the 775 orders without item
records, indicating that the absence of order-item data is strongly
associated with orders that were unavailable or canceled before completing
the normal order lifecycle.

The remaining cases consist of 5 `created`, 2 `invoiced`, and 1 `shipped`
orders.

Therefore, orders without item records should not be treated as general
data-quality errors or automatically removed from the raw dataset. Their
inclusion in downstream analysis should depend on the analytical objective.
For item-level sales analysis, only orders with corresponding item records
can contribute to product and revenue metrics.

#### Orders Without Payments

Only one order does not have a corresponding record in the `payments`
dataset.

Because this is an isolated case, inspect the order's identifier, customer,
status, and purchase timestamp to better understand the missing payment
relationship.

In [31]:
orders_without_payments = orders[
    ~orders["order_id"].isin(
        payments["order_id"]
    )
]

orders_without_payments[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ]
]

,order_id,customer_id,order_status,order_purchase_timestamp
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38


##### Orders Without Payments Findings

Only one order does not have a corresponding payment record.

The order (`bfbd0f9bdef84302105ad712db648a6c`) has a status of `delivered`,
indicating that the order progressed through the fulfillment process despite
having no matching record in the `payments` dataset.

Because this is a single isolated case, it should be retained rather than
removed automatically.

However, this order cannot contribute to payment-based metrics unless a
corresponding payment record is available. It should therefore be handled
carefully in analyses involving payment value, payment type, or payment
installments.

#### Orders Without Reviews

A total of 768 orders do not have corresponding records in the `reviews`
dataset.

Because review submission depends on both order completion and customer
behavior, examine the order-status distribution of these records to
determine whether missing reviews are associated with specific lifecycle
stages.

In [32]:
orders_without_reviews = orders[
    ~orders["order_id"].isin(
        reviews["order_id"]
    )
]

orders_without_reviews["order_status"].value_counts()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

##### Orders Without Reviews Findings

Among the 768 orders without corresponding review records, 646 have a
status of `delivered`.

The remaining orders are distributed across earlier or incomplete lifecycle
statuses, including 75 `shipped`, 20 `canceled`, 14 `unavailable`,
6 `processing`, 5 `invoiced`, and 2 `created` orders.

Unlike missing order-item records, the absence of review records is not
primarily associated with canceled or unavailable orders. Most orders
without reviews were successfully delivered.

Therefore, the absence of a review should not automatically be treated as
a data-quality error. Review records are not available for every order, and
orders without reviews should be retained in the dataset.

For analyses involving review scores or customer satisfaction, only orders
with corresponding review records can contribute to review-based metrics.

#### Order Relationship Coverage Findings

Relationship coverage is high across the core order-related datasets, but
not every order has corresponding item, payment, or review records.

- 98,666 of 99,441 orders (99.22%) have corresponding order-item records.
  Most orders without items are `unavailable` or `canceled`.
- 99,440 of 99,441 orders have corresponding payment records. The only order
  without a payment record has a status of `delivered`.
- 98,673 of 99,441 orders (99.23%) have corresponding review records. Most
  orders without reviews are `delivered`.

These findings indicate that missing relationships have different meanings
depending on the dataset. Missing order-item records are strongly associated
with order lifecycle status, while missing review records also occur
frequently among successfully delivered orders.

Therefore, orders should not be removed solely because a related record is
absent. Downstream analyses should instead define the appropriate analytical
population based on the required relationship and business question.

## 7. Data Type Assessment

Data types are reviewed across all datasets to determine whether each field
is stored in a format appropriate for its analytical purpose.

The assessment focuses on identifying:

- Date and timestamp fields stored as strings
- Identifier fields that should remain categorical or string-based
- ZIP code prefixes stored as numeric values
- Numeric fields affected by missing values
- Monetary and measurement fields that should remain numeric

No data types are modified in this notebook. Required conversions will be
performed during the data-cleaning stage.

In [33]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")

    dtype_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values
    })

    print(dtype_summary.to_string(index=False))


===== customers =====
                  column dtype
             customer_id   str
      customer_unique_id   str
customer_zip_code_prefix int64
           customer_city   str
          customer_state   str

===== geolocation =====
                     column   dtype
geolocation_zip_code_prefix   int64
            geolocation_lat float64
            geolocation_lng float64
           geolocation_city     str
          geolocation_state     str

===== order_items =====
             column   dtype
           order_id     str
      order_item_id   int64
         product_id     str
          seller_id     str
shipping_limit_date     str
              price float64
      freight_value float64

===== payments =====
              column   dtype
            order_id     str
  payment_sequential   int64
        payment_type     str
payment_installments   int64
       payment_value float64

===== reviews =====
                 column dtype
              review_id   str
               order_id 

### 7.1 Data Type Findings

The overall data types are appropriate for most identifier, categorical,
monetary, and measurement fields. However, several fields require conversion
before downstream analysis.

#### Date and Timestamp Fields

Eight date and timestamp fields are currently stored as strings:

- `order_items.shipping_limit_date`
- `reviews.review_creation_date`
- `reviews.review_answer_timestamp`
- `orders.order_purchase_timestamp`
- `orders.order_approved_at`
- `orders.order_delivered_carrier_date`
- `orders.order_delivered_customer_date`
- `orders.order_estimated_delivery_date`

These fields should be converted to datetime values during data cleaning to
support time-based analysis, including monthly sales trends, delivery
duration, and delivery-delay calculations.

#### ZIP Code Prefix Fields

The following ZIP code prefix fields are stored as integers:

- `customers.customer_zip_code_prefix`
- `geolocation.geolocation_zip_code_prefix`
- `sellers.seller_zip_code_prefix`

Although these fields contain numeric characters, they function as geographic
identifiers rather than quantities. They should therefore be converted to
string or categorical values during data cleaning.

#### Identifier and Categorical Fields

Customer, order, product, seller, review, payment-type, status, city, state,
and product-category fields are stored as strings, which is appropriate for
their analytical roles.

Integer sequence and score fields, such as `order_item_id`,
`payment_sequential`, `payment_installments`, and `review_score`, are also
appropriately stored as integers.

#### Product Attribute Fields

Several product attributes are stored as floating-point values, including
`product_name_lenght`, `product_description_lenght`, and
`product_photos_qty`.

These fields conceptually represent counts or lengths, but missing values
cause them to be represented as floating-point values. During data cleaning,
nullable integer types may be considered where appropriate.

Physical product measurements such as weight, length, height, and width
should remain numeric because they represent continuous quantities.

#### Monetary Fields

`price`, `freight_value`, and `payment_value` are stored as floating-point
values, which is appropriate for monetary analysis.

No data types are modified in this notebook. Required conversions will be
implemented in `02_data_cleaning.ipynb`.

## 8. Data Range and Validity Assessment

Numeric fields are examined to identify potentially invalid, unusual, or
extreme values that may affect downstream analysis.

Descriptive statistics are reviewed across transaction, payment, review,
product, and geolocation datasets to understand the overall distribution and
range of numeric variables.

The assessment focuses on:

- Zero or negative monetary values
- Invalid payment installment values
- Review scores outside the expected range
- Zero or implausible product measurements
- Extreme geographic coordinates
- Other unusual minimum or maximum values that require further investigation

Potential anomalies identified in this section are investigated before any
cleaning decisions are made.

In [34]:
numeric_checks = {
    "order_items": [
        "order_item_id",
        "price",
        "freight_value"
    ],
    "payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],
    "reviews": [
        "review_score"
    ],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "geolocation": [
        "geolocation_lat",
        "geolocation_lng"
    ]
}

for name, columns in numeric_checks.items():
    print(f"\n===== {name} =====")
    display(
        datasets[name][columns]
        .describe()
        .T
        .round(2)
    )


===== order_items =====


,count,mean,std,min,25%,50%,75%,max
order_item_id,112650.0,1.20,0.71,1.00,1.00,1.00,1.00,21.00
price,112650.0,120.65,183.63,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.99,15.81,0.00,13.08,16.26,21.15,409.68



===== payments =====


,count,mean,std,min,25%,50%,75%,max
payment_sequential,103886.0,1.09,0.71,1.0,1.00,1.0,1.00,29.00
payment_installments,103886.0,2.85,2.69,0.0,1.00,1.0,4.00,24.00
payment_value,103886.0,154.10,217.49,0.0,56.79,100.0,171.84,13664.08



===== reviews =====


,count,mean,std,min,25%,50%,75%,max
review_score,99224.0,4.09,1.35,1.0,4.0,5.0,5.0,5.0



===== products =====


,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.0,48.48,10.25,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,771.50,635.12,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.19,1.74,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.47,4282.04,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.82,16.91,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.94,13.64,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.20,12.08,6.0,15.0,20.0,30.0,118.0



===== geolocation =====


,count,mean,std,min,25%,50%,75%,max
geolocation_lat,1000163.0,-21.18,5.72,-36.61,-23.60,-22.92,-19.98,45.07
geolocation_lng,1000163.0,-46.39,4.27,-101.47,-48.57,-46.64,-43.77,121.11


### 8.1 Numeric Anomaly Screening

Based on the descriptive statistics, several numeric fields are screened for
values that may require further investigation.

The screening checks for:

- Zero product prices
- Zero freight values
- Zero payment installments
- Zero payment values
- Review scores outside the expected range of 1 to 5
- Zero product weights

These values are not automatically treated as data-quality errors.
Anomalies that may materially affect downstream analysis are investigated
in greater detail before cleaning decisions are made.

In [35]:
anomaly_counts = pd.DataFrame({
    "check": [
        "Zero price",
        "Zero freight",
        "Zero payment installments",
        "Zero payment value",
        "Invalid review score",
        "Zero product weight"
    ],
    "count": [
        (order_items["price"] == 0).sum(),
        (order_items["freight_value"] == 0).sum(),
        (payments["payment_installments"] == 0).sum(),
        (payments["payment_value"] == 0).sum(),
        (~reviews["review_score"].between(1, 5)).sum(),
        (products["product_weight_g"] == 0).sum()
    ]
})

anomaly_counts

,check,count
0,Zero price,0
1,Zero freight,383
2,Zero payment installments,2
3,Zero payment value,9
4,Invalid review score,0
5,Zero product weight,4


### 8.2 Payment Anomaly Investigation

The initial screening identifies two payment-related anomalies:

- 2 records have zero payment installments.
- 9 records have a payment value of zero.

To investigate these cases together, payment records meeting either
condition are extracted and compared by payment type, installment count,
and payment value.

These records are examined before determining whether they represent
data-quality issues or unusual but valid payment conditions.

In [36]:
zero_payment_cases = payments[
    (payments["payment_installments"] == 0)
    |
    (payments["payment_value"] == 0)
].copy()

zero_payment_cases.sort_values(
    ["payment_value", "payment_installments"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


#### Payment Anomaly Findings

The payment anomaly investigation identifies 11 records requiring attention.

Nine records have a `payment_value` of zero. Six of these records use
`voucher` as the payment type, while three are classified as `not_defined`.
All nine records have a `payment_installments` value of 1.

Two additional records have zero payment installments. Both use
`credit_card` as the payment type and contain positive payment values of
58.69 and 129.94.

The available data does not provide sufficient evidence to determine the
exact cause of these anomalies or to infer corrected values.

Therefore, these records should be retained and documented rather than
automatically corrected or removed. They should be handled carefully in
analyses involving payment value, payment type, or installment behavior.

### 8.3 Product Weight Investigation

The anomaly screening identifies four products with a recorded
`product_weight_g` value of zero.

Because a physical product is not expected to have zero weight, the affected
products are examined together with their other product attributes and
transaction history.

This investigation helps determine whether the zero-weight values represent
invalid measurements while preserving otherwise valid product and
transaction records.

In [37]:
zero_weight_products = products[
    products["product_weight_g"] == 0
]

zero_weight_products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [38]:
zero_weight_transactions = order_items[
    order_items["product_id"].isin(
        zero_weight_products["product_id"]
    )
]

print(
    "Zero-weight products:",
    len(zero_weight_products)
)

print(
    "Transactions involving zero-weight products:",
    len(zero_weight_transactions)
)

zero_weight_transactions[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "price",
        "freight_value"
    ]
]

Zero-weight products: 4
Transactions involving zero-weight products: 8


,order_id,order_item_id,product_id,price,freight_value
2972,06afc1144eb9f51ef2aa90ec9223c7f4,1,e673e90efa65a5409ff4196c038bb5af,129.9,23.71
2973,06afc1144eb9f51ef2aa90ec9223c7f4,2,e673e90efa65a5409ff4196c038bb5af,129.9,23.71
3052,06d9e69034388abf6da64378e10737b8,1,36ba42dd187055e1fbe943b2d11430ca,100.0,23.85
3053,06d9e69034388abf6da64378e10737b8,2,36ba42dd187055e1fbe943b2d11430ca,100.0,23.85
14080,200b121c28e10ef638131a7c76753327,1,81781c0fed9fe1ad6e8c81fca1e1cb08,100.0,19.89
31488,476b812a7e4fc972646eb390517bddcb,1,e673e90efa65a5409ff4196c038bb5af,129.9,23.71
32984,4abc7b5330425bcf9c2f7f48151a88c0,1,8038040ee2a71048d4bdbbdc985b69ab,129.9,14.49
79374,b489f7ae130ba3fd26b0a20f8cc81c61,1,e673e90efa65a5409ff4196c038bb5af,129.9,23.71


#### Product Weight Findings

Four products have a recorded `product_weight_g` value of zero.

All four products belong to the `cama_mesa_banho` category and contain
otherwise valid product attributes. Each product has dimensions of
30 × 25 × 30 cm and one product photo, indicating that the products
themselves are not empty or incomplete records.

These four products are also associated with 8 order-item transactions with
positive prices and freight values, confirming that they participated in
actual transactions.

Therefore, the zero-weight values are more likely to represent invalid or
missing product measurements rather than invalid products or transactions.

During data cleaning, the zero values in `product_weight_g` should be treated
as missing values rather than removing the affected products or their
transactions. No weight value should be imputed unless a defensible
estimation method is established.

### 8.4 Geolocation Coordinate Validation

The descriptive statistics reveal several extreme latitude and longitude
values in the `geolocation` dataset.

Because the dataset represents Brazilian locations, broad geographic bounds
are used as a sanity check to identify clearly unusual coordinate
observations.

The following approximate bounds are applied:

- Latitude: -35 to 6
- Longitude: -75 to -30

These bounds are used only to flag potentially invalid observations for
further investigation. They are not intended to represent the exact
geographic borders of Brazil.

In [39]:
geo_outliers = geolocation[
    (~geolocation["geolocation_lat"].between(-35, 6))
    |
    (~geolocation["geolocation_lng"].between(-75, -30))
].copy()

print(
    "Potential coordinate outliers:",
    len(geo_outliers)
)

print(
    "Percentage of geolocation records:",
    round(
        len(geo_outliers) / len(geolocation) * 100,
        4
    ),
    "%"
)

print(
    "ZIP code prefixes affected:",
    geo_outliers["geolocation_zip_code_prefix"].nunique()
)

geo_outliers[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].sort_values(
    ["geolocation_lat", "geolocation_lng"]
)

Potential coordinate outliers: 29
Percentage of geolocation records: 0.0029 %
ZIP code prefixes affected: 20


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
992584,98780,-36.605374,-64.283946,santa rosa,RS
993075,98780,-36.603837,-64.287433,santa rosa,RS
993302,98780,-36.603837,-64.287433,santa rosa,RS
965687,95130,14.585073,121.105394,santa lucia do piai,RS
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
698466,47310,38.268205,-7.803886,santana do sobrado,BA


#### Geolocation Coordinate Findings

The geographic sanity check identifies 29 potentially invalid coordinate
observations, representing only 0.0029% of the 1,000,163 records in the
`geolocation` dataset.

These observations affect 20 ZIP code prefixes. Several records contain
coordinates that are clearly inconsistent with the expected geographic
location of Brazilian addresses, including unusually large positive
latitudes and longitudes.

The identified cases represent individual geographic observations rather
than necessarily invalid ZIP code prefixes. As established in the
geolocation granularity analysis, each ZIP code prefix may contain multiple
latitude and longitude observations.

Therefore, entire ZIP code prefixes should not be removed because of a
small number of anomalous coordinates.

During data cleaning, clearly invalid coordinate observations will be
excluded before constructing a ZIP-code-level geographic representation.
The remaining valid observations can then be aggregated to obtain a
representative latitude and longitude for each ZIP code prefix.

The geographic bounds used here are broad sanity-check thresholds rather
than exact national borders and are intended only to identify clearly
unusual coordinate observations.

### 8.5 Data Range and Validity Assessment Summary

The numeric validity assessment identifies a small number of unusual values
that require different treatment depending on their business meaning.

Key findings include:

- No zero-priced order items were identified, and all recorded product
  prices are positive.
- 383 order-item records have zero freight values. These values may represent
  legitimate free-shipping transactions and should be retained unless
  additional evidence indicates otherwise.
- 2 credit-card payment records have zero installments despite having
  positive payment values. Because no supported replacement values can be
  inferred, these records should be retained and documented.
- 9 payment records have a payment value of zero, consisting of 6 `voucher`
  and 3 `not_defined` payment records. Their exact cause cannot be determined
  from the available data, so they should not be automatically removed or
  corrected.
- All review scores fall within the expected range of 1 to 5.
- 4 products have zero recorded weight but otherwise valid product attributes
  and are associated with 8 actual order-item transactions. Their zero
  weights should therefore be treated as missing measurements rather than
  invalid products.
- 29 geolocation observations fall outside the broad geographic sanity-check
  bounds, affecting 20 ZIP code prefixes and representing only 0.0029% of
  geolocation records. Clearly invalid coordinate observations should be
  excluded before ZIP-code-level aggregation.

Overall, the identified anomalies are limited in scope and do not justify
broad record deletion. Cleaning decisions should preserve valid transactions
while correcting or excluding only the specific fields or observations that
would otherwise affect downstream analysis.

## 9. Data Cleaning Plan

Based on the data-quality assessment conducted in the previous sections,
a cleaning plan is defined before any transformations are applied.

The purpose of this plan is to document how each identified issue will be
handled while preserving valid records and maintaining the original
relational structure of the dataset.

Cleaning decisions follow three general principles:

- Preserve valid transactions whenever possible.
- Avoid unsupported assumptions or arbitrary imputation.
- Apply transformations at the appropriate dataset grain to prevent
  unintended row loss or duplication.

The planned cleaning actions are summarized below.

| Dataset | Identified Issue | Planned Action |
|---|---|---|
| `customers` | ZIP code prefix stored as numeric | Convert `customer_zip_code_prefix` to string for identifier-based geographic joins. |
| `geolocation` | ZIP code prefix stored as numeric | Convert `geolocation_zip_code_prefix` to string. |
| `geolocation` | 261,831 exact duplicate records | Remove exact duplicate geographic observations. |
| `geolocation` | Multiple observations per ZIP code prefix | Create a ZIP-code-level geographic representation after cleaning individual observations. |
| `geolocation` | 29 potentially invalid coordinate observations | Exclude clearly invalid coordinate observations before ZIP-level aggregation. |
| `order_items` | `shipping_limit_date` stored as string | Convert to datetime. |
| `order_items` | 383 records with zero freight value | Retain because zero freight may represent legitimate shipping conditions and there is insufficient evidence to classify these records as invalid. |
| `payments` | Multiple payment records may exist per order | Preserve payment-level grain and aggregate to order level only when required by analysis. |
| `payments` | 9 zero-value payment records | Retain and document because the available data does not support correction or removal. |
| `payments` | 2 credit-card records with zero installments | Retain and document because no defensible replacement value can be inferred. |
| `reviews` | Review date fields stored as strings | Convert review date and timestamp fields to datetime. |
| `reviews` | Large number of missing comment titles and messages | Retain missing values because review comments are optional information. |
| `reviews` | Multiple review records may exist for an order | Preserve review-level records and aggregate carefully when order-level analysis is required. |
| `orders` | Lifecycle timestamp fields stored as strings | Convert order timestamp fields to datetime. |
| `orders` | Missing lifecycle timestamps | Preserve records and handle missing timestamps according to the requirements of each downstream metric. |
| `orders` | Delivered orders with missing lifecycle timestamps | Retain orders; exclude only from calculations that require the missing timestamp. |
| `products` | Missing category and descriptive attributes | Preserve valid product records; represent missing categories as unknown where categorical analysis requires a complete grouping. |
| `products` | 4 products with zero recorded weight | Convert zero `product_weight_g` values to missing values rather than removing the products. |
| `products` | Missing product measurements or attributes | Preserve products and transactions; avoid unsupported numerical imputation. |
| `sellers` | ZIP code prefix stored as numeric | Convert `seller_zip_code_prefix` to string for identifier-based geographic joins. |

These transformations will be implemented in `02_data_cleaning.ipynb`.
The raw datasets will remain unchanged so that the cleaning process is
reproducible and can be compared against the original source data.

## 10. Data Model and Analytical Grain

The Olist dataset consists of multiple relational tables operating at
different levels of granularity.

Understanding the analytical grain of each dataset is essential before
building cleaned analytical tables. Joining tables with incompatible grains
can unintentionally duplicate records and inflate business metrics such as
revenue, order count, payment value, or review count.

The core relationships can be represented as:

`customers` → `orders` → `order_items` → `products`

`order_items` → `sellers`

`orders` → `payments`

`orders` → `reviews`

`products` → `category_translation`

`customers` / `sellers` → `geolocation` through ZIP code prefixes after
geolocation data has been transformed to ZIP-code-level granularity.

### 10.1 Dataset Grain

Each dataset represents a different business entity or transactional level.

| Dataset | Analytical Grain | Key |
|---|---|---|
| `customers` | One order-level customer record | `customer_id` |
| `orders` | One order | `order_id` |
| `order_items` | One item within an order | (`order_id`, `order_item_id`) |
| `payments` | One payment record associated with an order | (`order_id`, `payment_sequential`) |
| `reviews` | One unique review-order relationship | (`review_id`, `order_id`) |
| `products` | One product | `product_id` |
| `sellers` | One seller | `seller_id` |
| `category_translation` | One product-category translation | `product_category_name` |
| `geolocation` | One geographic observation in the raw dataset | No single ZIP-level primary key |

The raw `geolocation` dataset does not operate at one-record-per-ZIP
granularity. A ZIP code prefix may contain multiple geographic observations,
so the dataset must be transformed before it is joined to customer or seller
records.

### 10.2 Customer Identifier Strategy

The customers dataset contains two different customer identifiers with
different analytical purposes.

`customer_id` uniquely identifies records in the `customers` dataset and is
the key used to connect customers with orders:

`orders.customer_id` → `customers.customer_id`

However, `customer_id` should not be used to measure the number of recurring
real-world customers.

The `customer_unique_id` field represents the persistent customer identifier
across multiple orders and should therefore be used for customer-level
analysis such as:

- Unique customer counts
- Repeat purchase behavior
- Purchase frequency
- Customer lifetime behavior
- RFM analysis

Therefore, `customer_id` will primarily be used for relational joins, while
`customer_unique_id` will be used for customer-level analytical metrics.

### 10.3 Join and Aggregation Considerations

The transactional datasets do not all operate at the same grain.

`order_items` may contain multiple records for one order because an order can
contain multiple products. Similarly, `payments` may contain multiple payment
records for one order, and `reviews` may contain multiple review records
associated with an order.

Therefore, directly joining raw `payments` or `reviews` records to an
item-level analytical table may create many-to-many row multiplication.

For example:

`orders` → `order_items`

creates an item-level dataset.

If raw payment records are then joined directly by `order_id`, an order with
multiple items and multiple payment records may produce duplicated
combinations of items and payments. This could incorrectly inflate metrics
such as payment value or transaction counts.

To prevent this:

- `order_items` will be used as the primary grain for product and item-level
  sales analysis.
- Payment data will be aggregated to the required analytical grain before
  being combined with item-level data when necessary.
- Review data will be aggregated or selected according to the required
  order-level analytical question before joining.
- Geolocation data will be transformed to ZIP-code-level granularity before
  being joined with customers or sellers.
- Customer-level analysis will use `customer_unique_id` after orders are
  connected to the customers dataset.

These rules will help prevent unintended row multiplication and preserve the
meaning of downstream business metrics.

### 10.4 Data Understanding Summary

The initial data-understanding process confirms that the Olist dataset has a
strong relational structure and generally high referential integrity, while
also containing several data-quality and granularity issues that require
careful handling.

The main considerations identified in this notebook include:

- Missing values in order lifecycle timestamps, product attributes, and
  optional review comments
- Exact duplicate observations and multiple geographic observations per ZIP
  code prefix in the geolocation dataset
- Different primary and composite key structures across transactional tables
- Distinct roles for `customer_id` and `customer_unique_id`
- A small number of payment, product-weight, and geographic anomalies
- Date fields requiring datetime conversion
- ZIP code prefixes requiring identifier-based data types
- Different analytical grains across order items, payments, reviews, and
  geolocation data

No broad record deletion is justified based on the findings from this
assessment. Instead, cleaning decisions will be applied selectively while
preserving valid products, orders, customers, and transactions.

The findings and cleaning plan established in this notebook provide the
foundation for `02_data_cleaning.ipynb`, where the required transformations
will be implemented before sales, customer, and delivery analyses are
performed.